In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


train_set = MNIST(root="./data", train=True, download=True, transform=transform)
test_set = MNIST(root="./data", train=False, download=True, transform=transform)

100%|██████████| 9.91M/9.91M [00:03<00:00, 2.60MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 96.1kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 793kB/s] 
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


In [31]:
train_set

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

In [32]:
test_set

Dataset MNIST
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

In [4]:
trainloader = DataLoader(train_set, batch_size=64, shuffle=True)
testloader = DataLoader(test_set, batch_size=64)

# Building the CNN

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier = CNN().to(device)
print(device)

cpu


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=0.001)

In [13]:
state_dict = torch.load("My_best_model.pth")
classifier.load_state_dict(state_dict)

#classifier.eval()

<All keys matched successfully>

In [9]:
# Once the model's best parameters are saved then no need to run this cell again 

epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        output = classifier(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=0.2529819905757904
epoch=2/10 & loss=0.07774508744478226
epoch=3/10 & loss=0.0551752932369709
epoch=4/10 & loss=0.04134397208690643
epoch=5/10 & loss=0.0366763137280941
epoch=6/10 & loss=0.02936111018061638
epoch=7/10 & loss=0.02541377954185009
epoch=8/10 & loss=0.02218002639710903
epoch=9/10 & loss=0.017866218462586403
epoch=10/10 & loss=0.018916571512818336


In [10]:
torch.save(classifier.state_dict(), "My_best_model.pth")

In [12]:
correct_labels = 0
total_labels = 0
all_preds = []
all_labels = []

classifier.eval()

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = classifier(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
print(f"Accuracy Score: {correct_labels / total_labels * 100} %")
print("Confusion Matrix: \n", confusion_matrix(all_labels, all_preds))

Accuracy Score: 98.6 %
Confusion Matrix: 
 [[ 971    1    2    1    0    0    3    2    0    0]
 [   0 1132    1    1    0    0    0    1    0    0]
 [   1    0 1015    4    0    0    2    8    1    1]
 [   0    0    2 1004    0    3    0    0    1    0]
 [   1    0    1    1  960    0    1    0    0   18]
 [   3    1    2    6    0  874    2    1    3    0]
 [   4    6    0    0    3    3  942    0    0    0]
 [   0    4    9    1    0    0    0 1012    2    0]
 [   2    1    3    4    1    1    0    0  959    3]
 [   3    1    1    1    3    2    0    6    1  991]]
